# Running Qiskit on IBM Quantum Hardware  
**Instructions (FYS5419/9419, Spring 2026)**

This notebook is a Jupyter-friendly version of the handout *Running Qiskit on IBM Quantum Hardware*.  
It walks through account setup, saving credentials, discovering backends, and running circuits using Qiskit Runtime primitives (Sampler/Estimator), including job monitoring and practical tips.


## What you will do

1. Create an IBM Quantum Platform / IBM Cloud account and select an access plan  
2. Create / select a **Qiskit Runtime** service instance  
3. Install Qiskit and IBM Runtime packages  
4. Save credentials locally (token / API key)  
5. Discover available backends (QPUs and simulators)  
6. Run circuits using Qiskit Runtime primitives (**Sampler / Estimator**)  
7. Monitor jobs and retrieve results


## Step 1: Create an account and choose a plan

- Create an account on the **IBM Quantum Platform** / **IBM Cloud**.  
- IBM Quantum Platform offers an *Open* (free) access path and paid plans (depending on region/org).  
- For QPU jobs through **Qiskit Runtime**, you typically need access to a Runtime instance linked to an IBM Cloud account.


## Step 2: Create / select a Qiskit Runtime service instance

- Log in to IBM Quantum Platform and ensure the correct account and region are selected.  
- Verify you have a **Qiskit Runtime service instance**; if not, create one.  
- You will connect to this instance from Python.


## Step 3: Install Qiskit and IBM Runtime packages

Use a virtual environment (recommended) and install Qiskit + IBM Runtime tooling.


In [ ]:
# (Terminal / shell) Create and activate a virtual environment, then install packages.
# Run these commands in a terminal, or prefix with '!' in a Jupyter cell if your environment allows it.

# python3 -m venv .venv
# source .venv/bin/activate
# pip install --upgrade pip
# pip install qiskit qiskit-ibm-runtime qiskit-ibm-provider


## Step 4: Save credentials locally (token / API key)

Obtain your API credentials (token/API key) from IBM Quantum Platform, then save them locally so you do not hard-code secrets in scripts.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Option A (common): save for later use (interactive)
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="YOUR_TOKEN_HERE",
    # instance may be required depending on your plan/org:
    # instance="ibm-q/open/main"  # example format; use your actual instance
    overwrite=True,
)

# Later in scripts/notebooks:
service = QiskitRuntimeService(channel="ibm_quantum_platform")


## Step 5: List available backends (QPUs and simulators)

Choose a backend based on availability, qubit count, and queue length.


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(channel="ibm_quantum_platform")

# List backends you can access
backends = service.backends()
print("Number of backends:", len(backends))

for b in backends[:10]:
    print(b.name)


## Why primitives (Sampler / Estimator)?

IBM recommends running circuits via **Qiskit Runtime primitives**:

- **Sampler**: returns samples / counts from circuit measurements  
- **Estimator**: returns expectation values of observables  

Primitives can include runtime features such as optimized execution and (optionally) error mitigation.


## Example A (Sampler): run a Bell circuit

This example creates a Bell state circuit, transpiles it for a target backend, and runs it using the Sampler primitive.


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService(channel="ibm_quantum_platform")

# Pick a backend (replace with one you have access to)
backend = service.backend("ibm_brisbane")  # example name

# Build circuit
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

# Transpile to the backend’s ISA
isa_circuit = transpile(qc, backend=backend, optimization_level=1)

# Run with the Sampler primitive
sampler = Sampler(backend=backend)
job = sampler.run([isa_circuit], shots=4000)

print("Job ID:", job.job_id())
result = job.result()
print(result)


## Example B (Estimator): expectation value of an observable

Estimator returns ⟨O⟩ for an observable O on the circuit output state.  
Typical use: VQE, QML, metrology objectives, etc.


In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_brisbane")  # example

# Prepare |Phi+> and estimate <ZZ>
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

obs = SparsePauliOp.from_list([("ZZ", 1.0)])

isa_circuit = transpile(qc, backend=backend, optimization_level=1)

estimator = Estimator(backend=backend)
job = estimator.run([(isa_circuit, obs)])

print("Job ID:", job.job_id())
res = job.result()
print(res)


## Sessions (recommended for multiple runs)

If you execute many primitive calls, using a **session** can reduce overhead by keeping context on the backend for a window of time (useful for iterative workflows).


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit import QuantumCircuit, transpile

service = QiskitRuntimeService(channel="ibm_quantum_platform")
backend = service.backend("ibm_brisbane")  # example

# Example circuit
qc = QuantumCircuit(2, 2)
qc.h(0); qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
isa_circuit = transpile(qc, backend=backend, optimization_level=1)

with Session(service=service, backend=backend) as session:
    sampler = Sampler(session=session)
    job = sampler.run([isa_circuit], shots=2000)
    print(job.result())


## Monitoring jobs and handling queues

Hardware runs are queued; queue times vary by backend load and access plan.  
Start with simulators, then validate on hardware. Keep shot counts modest during development.


In [ ]:
# For any job returned by Sampler/Estimator:
print("Status:", job.status())

# Blocking wait (simple):
result = job.result()
print(result)

# Some jobs expose logs/metadata depending on runtime configuration:
# print(job.metrics())  # if available in your environment


## Practical tips

- **Transpile matters:** always transpile to the backend target/ISA.  
- **Noise:** expect deviations from ideal counts/expectations; compare with a simulator baseline.  
- **Mitigation:** primitives may support resilience options depending on your plan.  
- **Reproducibility:** record backend name, calibration date/time, transpiler settings, and shots.  
- **Iteration:** develop locally (simulator), then run a small validation set on hardware.


## Checklist (quick workflow)

1. Create IBM Quantum Platform / IBM Cloud account and select an access plan  
2. Create/select a Qiskit Runtime instance for your account/region  
3. Install packages: `qiskit`, `qiskit-ibm-runtime`, `qiskit-ibm-provider`  
4. Save credentials with `QiskitRuntimeService.save_account(...)`  
5. List backends: `service.backends()`  
6. Build circuit, transpile to backend, run via Sampler or Estimator  
7. Retrieve results and document settings
